# Instructions
- Make a copy of this notebook in your drive.
- Load the **Netflix Movies and TV Shows dataset**.
- Read through each step and run the cells in order.
- Do not skip steps; each one builds on the previous one.
- Add your own observations wherever possible, especially when exploring graphs.
- This is practice for real-world data preprocessing and EDA, so think about *why* each step is done, not just *how*.
- At the end, feel free to explore further and add more plots, groupbys, or questions of your own.

Welcome to this checkpoint task.

Today we are exploring the Netflix Movies and TV Shows dataset to practice data preprocessing and exploratory data analysis (EDA). Think of the dataset as a big streaming catalogue that needs sorting before it is ready to be recommended to viewers - we will clean it, organize it, and look for patterns worth noticing.

We will be using pandas, numpy, matplotlib, and seaborn for this task.

Run the following cell to import them.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

In [ ]:
# We will use pandas for data handling,
# numpy for numerical operations,
# matplotlib and seaborn for visualizations.

## Setting the Scene

Before analyzing anything, we need to bring the dataset into our workspace.

**Your task:**
- Load the dataset into a pandas DataFrame.
- Look at the first few rows to see what the data contains.
- Check the shape of the dataset - how many titles and how many columns are we working with?

Think of this as opening the catalogue for the first time and getting a feel for its size.

In [ ]:
df = pd.read_csv("netflix_titles.csv")
df.head()

In [ ]:
df.shape

In [ ]:
print("Number of titles:", df.shape[0])
print("Number of columns:", df.shape[1])

### Observation
The dataset contains 6,234 titles and 12 columns before cleaning.

### Level 1: Know Your Columns

Every dataset has its own structure - some columns hold text, some hold numbers, and some may be incomplete.

**Your task:**
- Get a quick overview of column types and how many missing values each column has.
- Double-check that each column's data type makes sense for what it represents.

This is like reading the label on each shelf before you start organizing it.

In [ ]:
df.info()

### Observation
The `info()` output shows which columns are numeric/text-based and also shows missing values in several columns.

In [ ]:
df.isnull().sum()

### Level 2: Quick Stats Check

Before diving deeper, it helps to see a summary of the numerical columns.

**Your task:**
- Get summary statistics for the numerical columns.
- Note things like the earliest and most recent release years, and how the values are spread out.

This step gives you a first impression of the range and scale of the data.

In [ ]:
df.describe()

### Observation
The numerical columns include `release_year`; the summary gives its range and distribution before cleaning.

## Level 3: Clean the Catalogue (Missing Values)

Some entries in a streaming catalogue are incomplete, and we need to decide how to handle that.

**Your tasks:**
- Check how many missing values are in each column.
- For this dataset, handle missing values in **`director`** and **`country`** by removing those rows.
- Re-check to confirm there are no missing values left in those two columns.

> Tip: Do a quick sanity check after cleaning - the row count should drop a bit.

In [ ]:
missing_values = df.isnull().sum()
print(missing_values)

In [ ]:
df = df.dropna(subset=["director", "country"]).copy()

print("Shape after removing rows with missing director/country:", df.shape)

print("\nMissing values in director and country after cleaning:")
print(df[["director", "country"]].isnull().sum())

### Observation
Rows with missing `director` or `country` were removed as instructed. The cleaned dataset contains 4,094 rows, and both of those columns now have zero missing values.

## Level 4: Fix the `date_added` Column (Data Types)

The `date_added` column is usually stored as plain text, which makes it hard to work with dates properly.

**Your tasks:**
- Convert **`date_added`** to a proper datetime type.
- Create a new column called **`year_added`** that extracts just the year from `date_added`.
- Run a quick `info()` check to confirm the changes.

> If the conversion produces errors, check for stray spaces or unexpected formats in the column first.

In [ ]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")
df["year_added"] = df["date_added"].dt.year

df.info()

### Observation
`date_added` is now a datetime column, and `year_added` contains the year extracted from it. Some `date_added` values remain missing, so their `year_added` values are also missing.

## Level 5: The Longest Watches

Some titles run much longer than others. Let's find the standouts.

**Your task:**
- List the **Top 5 longest movies** in the dataset.
- Display their **title, country, release_year, and duration**.

Think of this as building a short list for viewers who want an epic movie night.

In [ ]:
movies = df[df["type"] == "Movie"].copy()

movies["duration_num"] = (
    movies["duration"]
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

top_5_longest = (
    movies.sort_values("duration_num", ascending=False)
    [["title", "country", "release_year", "duration"]]
    .head(5)
)

top_5_longest

### Observation
The longest movies in the cleaned dataset include *Sangam*, *Lagaan*, and *Jodhaa Akbar*.

## Level 6: Movies vs TV Shows

The catalogue contains two kinds of content: movies and TV shows.

**Your task:**
- Count how many titles fall into each `type` category.
- Create a bar chart to visualize the counts.
- Answer: which type makes up the larger share of the catalogue?

In [ ]:
type_counts = df["type"].value_counts()
type_counts

### Observation
Movies are much more common than TV shows in this cleaned dataset.

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(type_counts.index, type_counts.values)
plt.title("Movies vs TV Shows")
plt.xlabel("Type")
plt.ylabel("Number of Titles")
plt.show()

print("Observation:")
print(f"Movies make up the larger share of the catalogue with {type_counts.get('Movie', 0)} titles, compared with {type_counts.get('TV Show', 0)} TV shows.")

## Level 7: Where Does the Content Come From?

Netflix sources content from many countries.

**Your task:**
- Find the **Top 5 countries** with the highest number of titles.
- Show their contribution using a pie chart.

This will help us see which countries dominate the catalogue.

In [ ]:
country_counts = (
    df.assign(country=df["country"].str.split(", "))
      .explode("country")["country"]
      .value_counts()
      .head(5)
)

country_counts

### Observation
After splitting multi-country entries, the United States has the highest number of titles among the top five countries, followed by India.

In [ ]:
plt.figure(figsize=(7, 7))
plt.pie(
    country_counts.values,
    labels=country_counts.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Top 5 Countries by Number of Titles")
plt.show()

print("Observation:")
print(f"{country_counts.index[0]} contributes the largest number of titles among the top five countries.")

## Level 8: Genre Leaders by Country

Different genres may be led by different countries.

**Your task:**
- For each genre in `listed_in`, find the **country** that has produced the most titles in that genre.
- Print the results as a list (Genre -> Leading Country).

> Note: a title can belong to more than one genre - think about how you want to handle that before grouping.

In [ ]:
genre_country = (
    df.assign(
        genre=df["listed_in"].str.split(", "),
        country_name=df["country"].str.split(", ")
    )
    .explode("genre")
    .explode("country_name")
)

genre_country_counts = (
    genre_country
    .groupby(["genre", "country_name"])
    .size()
    .reset_index(name="count")
)

leader_indices = genre_country_counts.groupby("genre")["count"].idxmax()

genre_leaders = (
    genre_country_counts.loc[leader_indices]
    .sort_values("genre")
    .reset_index(drop=True)
)

for _, row in genre_leaders.iterrows():
    print(f"{row['genre']} -> {row['country_name']} ({row['count']} titles)")

### Observation
A title can have multiple genres and countries, so both columns were split and exploded before counting genre-country combinations. This lets each listed genre/country contribution be counted.

## Level 9: Netflix's Busiest Year

Netflix has grown its catalogue steadily over the years, but some years stand out more than others.

**Your task:**
- Group the data by `year_added`.
- Find the year in which Netflix added the **most titles** to its catalogue.

This shows us the point where the platform's library grew the fastest.

In [ ]:
year_counts = df.groupby("year_added").size().sort_values(ascending=False)

busiest_year = year_counts.idxmax()
busiest_count = year_counts.max()

print(f"Netflix added the most titles in {int(busiest_year)}.")
print(f"Number of titles added in that year: {int(busiest_count)}")

### Observation
2019 is the busiest year for titles added to Netflix in this cleaned dataset.

## Final Task: Open Exploration

You have worked through each step above - now it's time to explore on your own.

**Your final task:**
- Choose any 2 to 3 plots of your own choice that reveal an interesting pattern in the data.

This is your creative zone - treat it as building your own short story from the dataset. When you are done, share your best plot and the observation behind it with the group.

In [ ]:
# Plot 1: Number of titles added each year
plt.figure(figsize=(9, 5))
plt.plot(year_counts.sort_index().index, year_counts.sort_index().values, marker="o")
plt.title("Netflix Titles Added by Year")
plt.xlabel("Year Added")
plt.ylabel("Number of Titles")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Plot 2: Top 10 genres
genre_counts = (
    df["listed_in"]
    .str.split(", ")
    .explode()
    .value_counts()
    .head(10)
)

plt.figure(figsize=(9, 5))
plt.barh(genre_counts.index[::-1], genre_counts.values[::-1])
plt.title("Top 10 Genres")
plt.xlabel("Number of Titles")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

# Plot 3: Distribution of movie durations
plt.figure(figsize=(9, 5))
plt.hist(movies["duration_num"].dropna(), bins=25)
plt.title("Distribution of Movie Durations")
plt.xlabel("Duration (minutes)")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.show()

print("Observations:")
print("- The number of titles added rises strongly toward 2018-2019, with 2019 being the busiest year in this cleaned dataset.")
print("- The most common genres are dominated by broad categories such as International Movies, Dramas and Comedies.")
print("- Most movies are concentrated around typical feature-film lengths, while very long movies are much less common.")

### Note on the dataset
This notebook uses the `netflix_titles.csv` dataset supplied for this task. The analysis follows the task instructions: missing `director` and `country` rows are removed, `date_added` is converted to datetime, and multi-valued country/genre fields are split for the relevant analyses.